# r.noaa.atlas14

This notebook runs the examples from `r.noaa.atlas14`'s manual page and visualizes the
output. It demonstrates both acquisition modes:

- **mode=point** — query the NOAA Precipitation Frequency Data Server (PFDS) for a
  single longitude/latitude and render the intensity–duration–frequency table.
- **mode=grid** — download a NOAA Atlas 14 GIS-compatible grid ZIP for a specific
  region/ARI/duration and import it as a raster.

## Setup

We will be using the NC SPM sample project.

In [ ]:
import subprocess
import sys

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

gj.init("~/grassdata/nc_spm_full_v2alpha2/PERMANENT")

## Point mode: PFDS query for Raleigh, NC

Fetch the expected precipitation depth for downtown Raleigh and write both JSON
and a GRASS vector point map that carries the full tables as JSON attributes.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="point",
    coordinates="-78.6382,35.7796",
    statistic="depth",
    units="english",
    series="pds",
    bound="expected",
    format="json",
    output="/tmp/raleigh_atlas14.json",
    vector_output="atlas14_raleigh",
    overwrite=True,
)

In [ ]:
# Inspect the JSON payload
import json

with open("/tmp/raleigh_atlas14.json") as fh:
    data = json.load(fh)

print("Return periods (years):", data["table"]["return_periods_years"])
print("First few rows:")
for row in data["table"]["rows"][:5]:
    print(f"  {row['duration']:>8}  {row['values']}")

## Grid mode: import a single 100-year, 24-hour ZIP archive

This downloads one archive from NOAA's HDSC data directory and imports the
rescaled raster (inches) into the current mapset.

In [ ]:
gs.run_command(
    "r.noaa.atlas14",
    mode="grid",
    archive_url="https://hdsc.nws.noaa.gov/pub/hdsc/data/se/se100yr24ha.zip",
    output_prefix="a14",
    flags="i",
    overwrite=True,
)

In [ ]:
# Display the imported raster
imported = gs.list_grouped(type="raster", pattern="a14*")[gs.gisenv()["MAPSET"]]
raster_name = imported[0]
print("Displaying:", raster_name)

atlas_map = gj.Map()
atlas_map.d_rast(map=raster_name)
atlas_map.d_barscale()
atlas_map.show()